In [1]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response

In [4]:
tool_duckduckgo_search.invoke("What is the capital of France?")

'Paris[a] is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km 2 (40.7 sq mi), and a metropolitan population of 13.2 million as of January 2026. [4] Located on the river Seine in the centre of the Île-de-France region, it is the largest metropolitan area and fourth-most populous city in the European Union (EU). Nicknamed the "City ... Jun 29, 2018 · Paris is the capital city of France, as well as the country\'s largest city. 4 days ago · Paris is the capital of France (French Republic), situated in the Western Europe subregion of Europe. In Paris, the currency used is Euro (€), which is the official currency used in France. Learn why Paris is the capital of France and how it became a global city with a rich cultural heritage. Discover its geography, climate, population, landmarks, and industries. Nov 28, 2024 · Learn why Paris is the capital of France and how it became the political and cultural hub of the country. Discover

In [5]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about persons, places, etc. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response

In [6]:
tool_wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American businessman and entrepreneur who has been the chief executive officer (CEO) of the artificial intelligence research organization OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service, which raised more than $30 million in venture capital before being acquired by Green Dot Corporation for $43.4 million in cash. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. \nAfter co-founding OpenAI in 2015, Altman later became the organization\'s CEO in 2019. In 2023, he was ousted by the organization\'s board of directors, who cited a lack of "confidence in his ability to continue leading OpenAI" in an official post. The move was met with significant backlash from employees and investors, resulting in Altman\'s reins

In [7]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [8]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [9]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

In [10]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content

In [11]:
tool_rag.invoke("When was NovaSphere Organization founded?")

/var/folders/cr/34qx8njd48g07ps00m4fshq40000gn/T/ipykernel_22024/1495361009.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

## Bind Tools

In [12]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag
]

In [13]:
llm_bind = llm.bind_tools(toolkit)

In [14]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessar.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 316, 'total_tokens': 406, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzUx9lKJNZB2pIXWlCBCpMmcXvtR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0a9-3c39-7763-9f0f-3c6515efb4df-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_TWgRjYEtjRh1vaR1mwbbtndo', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 316, 'output_tokens': 90, 'total_tokens': 406, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

## ReAct Agent

In [15]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [16]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='ab5eae43-3ab8-4993-8810-88b892a155a7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 315, 'total_tokens': 405, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzVtTx7rlQdyvY9Dp9as7cubzUJJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0aa-1ff5-76a1-bf5b-bedb231365ca-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_RS8JrvWIce22Wfj1hf9VYjVj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

In [17]:
response = my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded?"}]}
)

## MEMORY

In [18]:
agent_memory = []

In [19]:
def invoke_agent_with_memory(query: str):

    response = my_agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer this query: {query}. You can refer to the previous interactions : {str(agent_memory)}"}]}
    )
    agent_memory.extend(response['messages'])
    return response

In [20]:
invoke_agent_with_memory("When was NovaSphere Organization founded?")

{'messages': [HumanMessage(content='Answer this query: When was NovaSphere Organization founded?. You can refer to the previous interactions : []', additional_kwargs={}, response_metadata={}, id='5e8dcd88-88a7-4798-8444-0cdb0254e378'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 322, 'total_tokens': 352, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzXJTee3Xjd4wlMCcpFC1WuxPQbK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0ab-75bb-7561-aa17-8c863784df1f-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_ilu8oWJgDbgW

In [21]:
invoke_agent_with_memory("Could you please summarize the previous interactions?")

{'messages': [HumanMessage(content='Answer this query: Could you please summarize the previous interactions?. You can refer to the previous interactions : [HumanMessage(content=\'Answer this query: When was NovaSphere Organization founded?. You can refer to the previous interactions : []\', additional_kwargs={}, response_metadata={}, id=\'5e8dcd88-88a7-4798-8444-0cdb0254e378\'), AIMessage(content=\'\', additional_kwargs={\'refusal\': None}, response_metadata={\'token_usage\': {\'completion_tokens\': 30, \'prompt_tokens\': 322, \'total_tokens\': 352, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_provider\': \'openai\', \'model_name\': \'gpt-5-mini-2025-08-07\', \'system_fingerprint\': None, \'id\': \'chatcmpl-DVzXJTee3Xjd4wlMCcpFC1WuxPQbK\', \'service_tier\': \'default\', \'finish_reason\': \'tool_calls\',

In [22]:
agent_memory

[HumanMessage(content='Answer this query: When was NovaSphere Organization founded?. You can refer to the previous interactions : []', additional_kwargs={}, response_metadata={}, id='5e8dcd88-88a7-4798-8444-0cdb0254e378'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 322, 'total_tokens': 352, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzXJTee3Xjd4wlMCcpFC1WuxPQbK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0ab-75bb-7561-aa17-8c863784df1f-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_ilu8oWJgDbgWHIZPEzkkRBfR',

In [23]:
while True:
    query = input("Ask me anything: ")
    response = invoke_agent_with_memory(query)
    print(response)
    flag = input("Do you want to continue? (yes/no): ")
    if flag.lower() == "no":
        break

{'messages': [HumanMessage(content='Answer this query: Could you please summarize the previous interactions?. You can refer to the previous interactions : [HumanMessage(content=\'Answer this query: When was NovaSphere Organization founded?. You can refer to the previous interactions : []\', additional_kwargs={}, response_metadata={}, id=\'5e8dcd88-88a7-4798-8444-0cdb0254e378\'), AIMessage(content=\'\', additional_kwargs={\'refusal\': None}, response_metadata={\'token_usage\': {\'completion_tokens\': 30, \'prompt_tokens\': 322, \'total_tokens\': 352, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_provider\': \'openai\', \'model_name\': \'gpt-5-mini-2025-08-07\', \'system_fingerprint\': None, \'id\': \'chatcmpl-DVzXJTee3Xjd4wlMCcpFC1WuxPQbK\', \'service_tier\': \'default\', \'finish_reason\': \'tool_calls\',